# Eurydice Prompt Evaluation Workbench

This notebook evaluates the current Eurydice music-agent prompt stack against the functionality that already exists in the codebase.

It is designed to help with:
- prompt engineering iteration
- diagnosing strengths and weaknesses
- checking reasoning discipline
- scoring feature coverage and scenario readiness

The workbench uses:
- `DEFAULT_MUSIC_SYSTEM_INSTRUCTIONS`
- the live-tool prompt fragment
- a representative live-context packet
- scenario-based heuristic scoring from `app.domains.music.prompt_eval`


In [1]:
from __future__ import annotations

import sys
from pathlib import Path
from types import SimpleNamespace
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "backend").exists():
    REPO_ROOT = REPO_ROOT.parent
BACKEND_ROOT = REPO_ROOT / "backend"
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from app.domains.music.constitution import DEFAULT_MUSIC_SYSTEM_INSTRUCTIONS
from app.domains.music.context import compose_live_context_packet
from app.domains.music.live_tools import music_live_tool_prompt_fragment
from app.domains.music.prompt_eval import DEFAULT_EVAL_SCENARIOS, evaluate_music_agent_prompt


def _clean(value: object) -> str:
    text = str(value)
    return text.replace("|", "/").replace("\n", " ").strip()


def markdown_table(rows: list[dict[str, object]], headers: list[str]) -> str:
    if not rows:
        return "_No rows._"
    header_line = "| " + " | ".join(headers) + " |"
    divider = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(_clean(row.get(header, "")) for header in headers) + " |")
    return "\n".join([header_line, divider, *body])


def scorecard(report: dict[str, object]) -> list[dict[str, object]]:
    return [
        {"Metric": "Aggregate", "Score": report["aggregate_score"]},
        {"Metric": "Prompt quality", "Score": report["prompt_quality_score"]},
        {"Metric": "Reasoning", "Score": report["reasoning_score"]},
        {"Metric": "Functionality", "Score": report["functionality_score"]},
        {"Metric": "Scenario readiness", "Score": report["scenario_score"]},
    ]


In [2]:
sample_profile = SimpleNamespace(
    weakest_dimension="rhythm",
    instrument_profile="PIANO",
    overall_score=0.72,
    consistency_score=0.61,
    practice_frequency=0.57,
    sample_count=11,
)

sample_attempts = [
    SimpleNamespace(
        measure_index=2,
        accuracy=0.68,
        match=False,
        needs_replay=True,
        summary="Dragged the last eighth note and lost bar alignment.",
    ),
    SimpleNamespace(
        measure_index=3,
        accuracy=0.83,
        match=True,
        needs_replay=False,
        summary="Pitch secure; tempo stabilized after the second beat.",
    ),
]

sample_library_items = [
    SimpleNamespace(
        title="Subdivision Ladder in 4/4",
        content_type="exercise",
        difficulty="intermediate",
        instrument="PIANO",
        technique_tags=["rhythm", "subdivision", "timing"],
        learning_objective="Stabilize beat placement while switching subdivisions.",
    ),
    SimpleNamespace(
        title="Two-bar syncopation drill",
        content_type="exercise",
        difficulty="intermediate",
        instrument="PIANO",
        technique_tags=["rhythm", "groove"],
        learning_objective="Improve off-beat attacks without losing tempo center.",
    ),
]

tool_fragment = music_live_tool_prompt_fragment()
context_packet = compose_live_context_packet(
    skill="COMPARE_PERFORMANCE",
    goal="Coach one prepared bar at a time and explain rhythm mismatches precisely.",
    profile=sample_profile,
    attempts=sample_attempts,
    library_items=sample_library_items,
    max_chars=2200,
)

baseline_prompt = DEFAULT_MUSIC_SYSTEM_INSTRUCTIONS
baseline_report = evaluate_music_agent_prompt(
    baseline_prompt,
    tool_prompt_fragment=tool_fragment,
    context_policy_fragment=context_packet,
)

display(Markdown("## Baseline Scorecard"))
display(Markdown(markdown_table(scorecard(baseline_report), ["Metric", "Score"])))


## Baseline Scorecard

| Metric | Score |
| --- | --- |
| Aggregate | 1.0 |
| Prompt quality | 1.0 |
| Reasoning | 1.0 |
| Functionality | 1.0 |
| Scenario readiness | 1.0 |

In [3]:
display(Markdown("## Strengths"))
for item in baseline_report["strengths"]:
    display(Markdown(f"- {item}"))

display(Markdown("## Weaknesses"))
if baseline_report["weaknesses"]:
    for item in baseline_report["weaknesses"]:
        display(Markdown(f"- {item}"))
else:
    display(Markdown("- No heuristic coverage gaps detected for the current rubric."))


## Strengths

- Coach the user to frame a clear score region.

- Read visible notation from a score.

- Identify a melody, interval, chord, or arpeggio from live audio.

- Compare a performed phrase against an intended phrase or score.

...

- Require verification before final musical claims.

- Request narrower retries when evidence is noisy or unresolved.

- Treat stored context as supporting material and prioritize live evidence.

## Weaknesses

- No heuristic coverage gaps detected for the current rubric.

In [4]:
capability_rows = [
    {
        "Capability": check["key"],
        "Category": check["category"],
        "Passed": check["passed"],
        "Matched marker": check["matched_marker"] or "-",
        "Description": check["description"],
    }
    for check in baseline_report["capabilities"]["checks"]
]

display(Markdown("## Capability Coverage"))
display(Markdown(markdown_table(
    capability_rows,
    ["Capability", "Category", "Passed", "Matched marker", "Description"],
)))


## Capability Coverage

| Capability | Category | Passed | Matched marker | Description |
| --- | --- | --- | --- | --- |
| sheet_frame_coach | functionality | True | SHEET_FRAME_COACH | Coach the user to frame a clear score region. |
| read_score | functionality | True | READ_SCORE | Read visible notation from a score. |
| hear_phrase | functionality | True | HEAR_PHRASE | Identify a melody, interval, chord, or arpeggio from live audio. |
| compare_performance | functionality | True | COMPARE_PERFORMANCE | Compare a performed phrase against an intended phrase or score. |
| ear_train | functionality | True | EAR_TRAIN | Run a focused ear-training drill. |
| generate_example | functionality | True | GENERATE_EXAMPLE | Generate a labelled original phrase or exercise. |
| guided_lesson_tools | functionality | True | "name":"lesson_action" | Use deterministic lesson tools when structured score or lesson data is needed. |
| no_guessing | reasoning | True | Never guess | Refuse to guess uncertain musical facts. |
| single_step_guidance | reasoning | True | one musical instruction | Give only one musical instruction at a time. |
| confidence_signaling | reasoning | True | If confidence is limited | Expose confidence limits and uncertainty explicitly. |
| observation_vs_inference | reasoning | True | what you heard, what you inferred, and what still needs verification | Separate observation, inference, and verification state. |
| verification_loop | reasoning | True | Verify before claiming | Require verification before final musical claims. |
| replay_recovery | reasoning | True | ask for a replay | Request narrower retries when evidence is noisy or unresolved. |
| live_evidence_priority | reasoning | True | Prioritize live audio/video evidence | Treat stored context as supporting material and prioritize live evidence. |

In [5]:
scenario_rows = [
    {
        "Scenario": scenario["title"],
        "Score": scenario["score"],
        "Missing functionality": ", ".join(scenario["missing_required"]) or "-",
        "Missing reasoning": ", ".join(scenario["missing_reasoning"]) or "-",
        "User request": scenario["user_request"],
    }
    for scenario in baseline_report["scenarios"]["scenarios"]
]

display(Markdown("## Scenario Readiness"))
display(Markdown(markdown_table(
    scenario_rows,
    ["Scenario", "Score", "Missing functionality", "Missing reasoning", "User request"],
)))

display(Markdown("## Current Scenario Set"))
for scenario in DEFAULT_EVAL_SCENARIOS:
    display(Markdown(f"- **{scenario.title}**: {scenario.user_request}"))


## Scenario Readiness

| Scenario | Score | Missing functionality | Missing reasoning | User request |
| --- | --- | --- | --- | --- |
| Camera-based score reading | 1.0 | - | - | Read one visible bar from my score and tell me what is on it. |
| Live phrase identification | 1.0 | - | - | I just played four notes. Identify the arpeggio without guessing. |
| Score-to-performance comparison | 1.0 | - | - | Compare my take against the current bar and explain the mismatch. |
| One-step ear training drill | 1.0 | - | - | Give me one interval drill and verify my answer after I respond. |
| Generated exercise phrase | 1.0 | - | - | Create a short exercise phrase for me to practice and mark it as generated. |

## Current Scenario Set

- **Camera-based score reading**: Read one visible bar from my score and tell me what is on it.

- **Live phrase identification**: I just played four notes. Identify the arpeggio without guessing.

- **Score-to-performance comparison**: Compare my take against the current bar and explain the mismatch.

- **One-step ear training drill**: Give me one interval drill and verify my answer after I respond.

- **Generated exercise phrase**: Create a short exercise phrase for me to practice and mark it as generated.

## Candidate Prompt Comparison

Edit the candidate prompt below to test prompt revisions before you ship them.

Suggested workflow:
1. Edit `candidate_prompt`.
2. Re-run the comparison cell.
3. Check whether the score moved and whether strengths/weaknesses changed in the right direction.


In [6]:
candidate_prompt = DEFAULT_MUSIC_SYSTEM_INSTRUCTIONS
candidate_tool_fragment = tool_fragment
candidate_context_fragment = context_packet

candidate_report = evaluate_music_agent_prompt(
    candidate_prompt,
    tool_prompt_fragment=candidate_tool_fragment,
    context_policy_fragment=candidate_context_fragment,
)

comparison_rows = []
for baseline_row, candidate_row in zip(scorecard(baseline_report), scorecard(candidate_report)):
    delta = round(float(candidate_row["Score"]) - float(baseline_row["Score"]), 3)
    comparison_rows.append(
        {
            "Metric": baseline_row["Metric"],
            "Baseline": baseline_row["Score"],
            "Candidate": candidate_row["Score"],
            "Delta": delta,
        }
    )

display(Markdown("## Baseline vs Candidate"))
display(Markdown(markdown_table(comparison_rows, ["Metric", "Baseline", "Candidate", "Delta"])))

strengths_gained = sorted(set(candidate_report["strengths"]) - set(baseline_report["strengths"]))
new_weaknesses = sorted(set(candidate_report["weaknesses"]) - set(baseline_report["weaknesses"]))

display(Markdown("## Candidate Diagnosis"))
if strengths_gained:
    display(Markdown("**New strengths**"))
    for item in strengths_gained:
        display(Markdown(f"- {item}"))
else:
    display(Markdown("- No new strengths relative to baseline."))

if new_weaknesses:
    display(Markdown("**New weaknesses**"))
    for item in new_weaknesses:
        display(Markdown(f"- {item}"))
else:
    display(Markdown("- No new weaknesses relative to baseline."))


## Baseline vs Candidate

| Metric | Baseline | Candidate | Delta |
| --- | --- | --- | --- |
| Aggregate | 1.0 | 1.0 | 0.0 |
| Prompt quality | 1.0 | 1.0 | 0.0 |
| Reasoning | 1.0 | 1.0 | 0.0 |
| Functionality | 1.0 | 1.0 | 0.0 |
| Scenario readiness | 1.0 | 1.0 | 0.0 |

## Candidate Diagnosis

- No new strengths relative to baseline.

- No new weaknesses relative to baseline.

In [7]:
display(Markdown("## Prompt Bundle Preview"))
display(Markdown("### System prompt"))
print(baseline_prompt)

display(Markdown("### Tool prompt fragment"))
print(tool_fragment)

display(Markdown("### Context packet"))
print(context_packet)


## Prompt Bundle Preview

### System prompt

You are Eurydice, a precise real-time music tutor and analysis assistant.

Non-negotiable rules:
- Never guess notes, rhythm, chord quality, fingering, or notation. If uncertain, say what is uncertain and ask for a replay or a better score view.
- Give one musical instruction or one correction at a time.
- Prefer structured, teacher-like feedback over vague praise.
- Distinguish clearly between what you heard, what you inferred, and what still needs verification.
- If audio is noisy, clipped, or too polyphonic to resolve confidently, say so and ask for a simpler replay.

Core music tasks:
- SHEET_FRAME_COACH: coach the user to frame one stave or one short score region clearly.
- READ_SCORE: describe visible notation, one measure group at a time.
- HEAR_PHRASE: identify a melody, chord, interval, or arpeggio with confidence notes.
- COMPARE_PERFORMANCE: compare what was played against the intended notes or rhythm and explain the mismatch.
- EAR_TRAIN: run one listening drill at a time, 

### Tool prompt fragment

If you need deterministic score/lesson data, request a tool call by emitting exactly one line:
TOOL_CALL: {"name":"lesson_action","args":{...}}
Supported tool names: lesson_action, lesson_step, render_score.
Do not include extra prose in a TOOL_CALL line.


### Context packet

SESSION_SKILL: COMPARE_PERFORMANCE
GOAL: Coach one prepared bar at a time and explain rhythm mismatches precisely.
PROFILE: weakest_dimension=rhythm; instrument=PIANO; overall=72%; consistency=61%; practice_frequency=57%; samples=11
RECENT_ATTEMPTS:
- measure=2; accuracy=68%; match=no; replay=yes; summary=Dragged the last eighth note and lost bar alignment.
- measure=3; accuracy=83%; match=yes; replay=no; summary=Pitch secure; tempo stabilized after the second beat.
RELEVANT_LIBRARY_ITEMS:
- title=Subdivision Ladder in 4/4; type=exercise; difficulty=intermediate; instrument=PIANO; tags=rhythm,subdivision,timing; objective=Stabilize beat placement while switching subdivisions.
- title=Two-bar syncopation drill; type=exercise; difficulty=intermediate; instrument=PIANO; tags=rhythm,groove; objective=Improve off-beat attacks without losing tempo center.
CONTEXT_POLICY: Treat this as supporting context only. Prioritize live audio/video evidence. If evidence is unclear or conflicts, request 